# Candidate Sampling

This notebook constructs candidate samples for the manual evaluation of
MegaDetector predictions on MammalWeb images.

Sampling is conducted separately for:

- human detections;
- animal detections;
- vehicle detections.

The manual annotations produced from these samples will later be used as the
reference labels for threshold sensitivity analysis. MegaDetector confidence
and `contains_human` are not treated as ground-truth labels.

The sampling procedure aims to provide coverage across:

- MegaDetector confidence;
- time of day;
- geographic location;
- season;
- sites;
- camera-trap sequences;
- relative bounding-box size, once image dimensions have been obtained.

Repeated frames from the same sequence are restricted because they are unlikely
to represent statistically independent observations.

## Sample-size considerations

There is no universal sample size that is suitable for every classifier
evaluation. The required number of annotations depends on:

- the prevalence of the target class;
- the desired precision of the performance estimates;
- the number of subgroup comparisons;
- the number of statistically independent observations;
- the annotation time available.

Beleites et al. (2012) discuss the importance of having sufficient independent
test observations and show that model validation may require substantially more
cases than model development, particularly when comparing classifiers or
estimating performance precisely.

Poms et al. (2021) demonstrate that model confidence scores and importance
sampling can reduce the number of labels required when evaluating rare
categories. However, such non-random sampling requires appropriate statistical
weighting.

For this project, a stratified candidate pool will first be produced. A pilot
subset will be annotated to estimate:

- annotation speed;
- the prevalence of true objects within each confidence band;
- the frequency of difficult or ambiguous cases;
- the uncertainty of the resulting performance estimates.

The final annotation target may then be adjusted using the pilot results.

References:

- Beleites, C. et al. (2012). Sample Size Planning for Classification Models.
  arXiv:1211.1323.
- Poms, F. et al. (2021). Low-Shot Validation: Active Importance Sampling for
  Estimating Classifier Performance on Rare Categories. arXiv:2109.05720.

## Initial sampling proposal

The initial design uses an oversized candidate pool followed by a smaller final
manual sample.

For positive-confidence detections:

- primary sampling cells: confidence band × time window;
- 40 candidate images per cell;
- provisional final target of 20 images per cell;
- maximum one selected image from each sequence;
- maximum two candidate images from each site;
- geographic group and season used as balancing variables.

There are seven positive-confidence bands and four usable time windows:

- 28 positive sampling cells per target class;
- up to 1,120 positive candidates per class;
- provisional final target of 560 positive images per class.

Zero-confidence sampling will be handled separately because:

- zero-confidence rows require different treatment from detected objects;
- relative bounding-box size is not applicable;
- some zero-confidence rows currently lack supplementary metadata;
- true target objects may be rare within the zero-confidence population.

The numbers above are provisional and may be revised following pilot
annotation.

In [1]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

In [2]:
INPUT_PATH = Path("sampling_prepared.csv")
OUTPUT_DIR = Path("sampling_outputs")

OUTPUT_DIR.mkdir(exist_ok=True)

print("Input file exists:", INPUT_PATH.exists())
print("Output directory:", OUTPUT_DIR)

Input file exists: True
Output directory: sampling_outputs


In [3]:
df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["taken_dt"],
    low_memory=False
)

print("Dataset loaded")
print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])

Dataset loaded
Rows: 6,720,367
Columns: 64


## Sampling configuration

Positive-confidence candidate sampling is performed separately for human,
animal and vehicle detections.

The primary sampling cells are defined by confidence band and time window.
Geographic group and season are used as balancing variables rather than
additional formal strata.

In [4]:
RANDOM_SEED = 42

TARGET_CLASSES = [
    "human",
    "animal",
    "vehicle"
]

POSITIVE_CONFIDENCE_BANDS = [
    "0.01_to_0.02",
    "0.02_to_0.05",
    "0.05_to_0.10",
    "0.10_to_0.30",
    "0.30_to_0.70",
    "0.70_to_0.90",
    "0.90_to_1.00"
]

USABLE_TIME_WINDOWS = [
    "overnight",
    "morning_transition",
    "daytime_core",
    "evening_transition"
]

CANDIDATES_PER_CELL = 40
PROVISIONAL_FINAL_PER_CELL = 20

CANDIDATE_SITE_CAP = 2
FINAL_SITE_CAP = 1
SEQUENCE_CAP = 1

In [5]:
sampling_columns = {
    "human": {
        "score": "mega_human_confidence",
        "band": "human_confidence_band",
        "bbox_probability": "human_bbox_prob",
        "bbox_area": "human_bbox_area_px",
        "has_bbox": "human_has_bbox"
    },
    "animal": {
        "score": "mega_animal_confidence",
        "band": "animal_confidence_band",
        "bbox_probability": "animal_bbox_prob",
        "bbox_area": "animal_bbox_area_px",
        "has_bbox": "animal_has_bbox"
    },
    "vehicle": {
        "score": "mega_vehicle_confidence",
        "band": "vehicle_confidence_band",
        "bbox_probability": "vehicle_bbox_prob",
        "bbox_area": "vehicle_bbox_area_px",
        "has_bbox": "vehicle_has_bbox"
    }
}

## Positive candidate eligibility

Candidate eligibility is assessed separately for human, animal and vehicle
detections.

A photograph is eligible for the positive-confidence candidate pool when:

- the class confidence is greater than zero;
- its confidence band is one of the seven positive bands;
- usable time metadata is available;
- geographic and site metadata are available;
- sequence information is available;
- the relevant MegaDetector bounding box is available and valid;
- the image path information required for later S3 access is available.

No photographs are sampled during this step. Boolean eligibility masks are
created so that exclusion counts can be inspected before candidate selection.

In [6]:
positive_eligibility_masks = {}
eligibility_summaries = []

for target_class in TARGET_CLASSES:
    score_column = sampling_columns[target_class]["score"]
    band_column = sampling_columns[target_class]["band"]
    bbox_area_column = sampling_columns[target_class]["bbox_area"]
    has_bbox_column = sampling_columns[target_class]["has_bbox"]

    positive_confidence = (
        df[score_column].gt(0)
        & df[band_column].isin(POSITIVE_CONFIDENCE_BANDS)
    )

    usable_time = (
        df["has_time_metadata"].fillna(False)
        & df["time_window"].isin(USABLE_TIME_WINDOWS)
        & df["season"].notna()
    )

    usable_geography = (
        df["has_geographic_metadata"].fillna(False)
        & df["site_id"].notna()
        & df["geographic_group"].notna()
    )

    usable_sequence = (
        df["has_sequence_metadata"].fillna(False)
        & df["sequence_id"].notna()
    )

    usable_image_path = (
        df["filename"].notna()
        & df["dirname"].notna()
    )

    valid_bbox = (
        df[has_bbox_column].fillna(False)
        & df[bbox_area_column].notna()
        & df[bbox_area_column].gt(0)
    )

    after_time = (
        positive_confidence
        & usable_time
    )

    after_geography = (
        after_time
        & usable_geography
    )

    after_sequence = (
        after_geography
        & usable_sequence
    )

    after_image_path = (
        after_sequence
        & usable_image_path
    )

    eligible = (
        after_image_path
        & valid_bbox
    )

    positive_eligibility_masks[target_class] = eligible

    eligibility_summaries.append({
        "target_class": target_class,
        "positive_images": int(positive_confidence.sum()),
        "after_time_requirement": int(after_time.sum()),
        "after_geography_requirement": int(after_geography.sum()),
        "after_sequence_requirement": int(after_sequence.sum()),
        "after_image_path_requirement": int(after_image_path.sum()),
        "final_eligible_images": int(eligible.sum()),
        "total_excluded": int(
            positive_confidence.sum() - eligible.sum()
        )
    })

eligibility_summary = pd.DataFrame(eligibility_summaries)

display(eligibility_summary)

,target_class,positive_images,after_time_requirement,after_geography_requirement,after_sequence_requirement,after_image_path_requirement,final_eligible_images,total_excluded
0,human,614560,614551,614551,614551,614551,614551,9
1,animal,5579661,5579229,5579229,5579229,5579229,5579229,432
2,vehicle,679937,679902,679902,679902,679902,679902,35


In [7]:
eligibility_summary["eligible_percentage"] = (
    eligibility_summary["final_eligible_images"]
    / eligibility_summary["positive_images"]
    * 100
)

eligibility_summary["excluded_percentage"] = (
    eligibility_summary["total_excluded"]
    / eligibility_summary["positive_images"]
    * 100
)

display(eligibility_summary)

,target_class,positive_images,after_time_requirement,after_geography_requirement,after_sequence_requirement,after_image_path_requirement,final_eligible_images,total_excluded,eligible_percentage,excluded_percentage
0,human,614560,614551,614551,614551,614551,614551,9,99.998536,0.001464
1,animal,5579661,5579229,5579229,5579229,5579229,5579229,432,99.992258,0.007742
2,vehicle,679937,679902,679902,679902,679902,679902,35,99.994852,0.005148


In [8]:
assert all(
    len(mask) == len(df)
    for mask in positive_eligibility_masks.values()
)

In [9]:
exclusion_reason_summaries = []

for target_class in TARGET_CLASSES:
    score_column = sampling_columns[target_class]["score"]
    band_column = sampling_columns[target_class]["band"]
    bbox_area_column = sampling_columns[target_class]["bbox_area"]
    has_bbox_column = sampling_columns[target_class]["has_bbox"]

    positive_confidence = (
        df[score_column].gt(0)
        & df[band_column].isin(POSITIVE_CONFIDENCE_BANDS)
    )

    exclusion_reason_summaries.append({
        "target_class": target_class,
        "missing_or_unusable_time": int(
            (
                positive_confidence
                & (
                    ~df["has_time_metadata"].fillna(False)
                    | ~df["time_window"].isin(USABLE_TIME_WINDOWS)
                    | df["season"].isna()
                )
            ).sum()
        ),
        "missing_geography_or_site": int(
            (
                positive_confidence
                & (
                    ~df["has_geographic_metadata"].fillna(False)
                    | df["site_id"].isna()
                    | df["geographic_group"].isna()
                )
            ).sum()
        ),
        "missing_sequence": int(
            (
                positive_confidence
                & (
                    ~df["has_sequence_metadata"].fillna(False)
                    | df["sequence_id"].isna()
                )
            ).sum()
        ),
        "missing_image_path": int(
            (
                positive_confidence
                & (
                    df["filename"].isna()
                    | df["dirname"].isna()
                )
            ).sum()
        ),
        "missing_or_invalid_bbox": int(
            (
                positive_confidence
                & (
                    ~df[has_bbox_column].fillna(False)
                    | df[bbox_area_column].isna()
                    | ~df[bbox_area_column].gt(0)
                )
            ).sum()
        )
    })

exclusion_reason_summary = pd.DataFrame(
    exclusion_reason_summaries
)

display(exclusion_reason_summary)

,target_class,missing_or_unusable_time,missing_geography_or_site,missing_sequence,missing_image_path,missing_or_invalid_bbox
0,human,9,7,0,0,7
1,animal,432,197,0,0,197
2,vehicle,35,15,0,0,15


## Positive sampling-cell availability

The eligible positive-confidence population is examined across the primary
sampling cells defined by confidence band and time window.

For each target class and sampling cell, the following are calculated:

- number of eligible photographs;
- number of unique sequences;
- number of unique sites;
- number of geographic groups represented;
- number of seasons represented.

These diagnostics determine whether the proposed quota of 40 candidate images
per cell can be achieved while applying a maximum of one image per sequence
and two candidate images per site.

In [10]:
cell_availability_tables = {}

for target_class in TARGET_CLASSES:
    band_column = sampling_columns[target_class]["band"]
    eligibility_mask = positive_eligibility_masks[target_class]

    eligible_columns = [
        band_column,
        "time_window",
        "photo_id",
        "sequence_id",
        "site_id",
        "geographic_group",
        "season"
    ]

    eligible_data = df.loc[
        eligibility_mask,
        eligible_columns
    ]

    cell_table = (
        eligible_data
        .groupby(
            [band_column, "time_window"],
            observed=True
        )
        .agg(
            eligible_images=("photo_id", "size"),
            unique_sequences=("sequence_id", "nunique"),
            unique_sites=("site_id", "nunique"),
            geographic_groups=("geographic_group", "nunique"),
            seasons=("season", "nunique")
        )
        .reset_index()
    )

    cell_table["site_cap_capacity"] = (
        cell_table["unique_sites"]
        * CANDIDATE_SITE_CAP
    )

    cell_table["estimated_candidate_capacity"] = cell_table[
        [
            "eligible_images",
            "unique_sequences",
            "site_cap_capacity"
        ]
    ].min(axis=1)

    cell_table["meets_candidate_target"] = (
        cell_table["estimated_candidate_capacity"]
        >= CANDIDATES_PER_CELL
    )

    cell_availability_tables[target_class] = cell_table

    print(target_class.upper())
    display(cell_table)

HUMAN


,human_confidence_band,time_window,eligible_images,unique_sequences,unique_sites,geographic_groups,seasons,site_cap_capacity,estimated_candidate_capacity,meets_candidate_target
0,0.01_to_0.02,daytime_core,123172,85114,1800,5,4,3600,3600,True
1,0.01_to_0.02,evening_transition,21824,16323,1185,5,4,2370,2370,True
2,0.01_to_0.02,morning_transition,17138,12703,1106,5,4,2212,2212,True
3,0.01_to_0.02,overnight,8654,6960,580,5,4,1160,1160,True
4,0.02_to_0.05,daytime_core,115230,76904,1759,5,4,3518,3518,True
5,0.02_to_0.05,evening_transition,20974,14961,1147,5,4,2294,2294,True
6,0.02_to_0.05,morning_transition,16718,11794,1057,5,4,2114,2114,True
7,0.02_to_0.05,overnight,7519,5757,526,5,4,1052,1052,True
8,0.05_to_0.10,daytime_core,57437,43112,1542,5,4,3084,3084,True
9,0.05_to_0.10,evening_transition,10952,8487,919,5,4,1838,1838,True


ANIMAL


,animal_confidence_band,time_window,eligible_images,unique_sequences,unique_sites,geographic_groups,seasons,site_cap_capacity,estimated_candidate_capacity,meets_candidate_target
0,0.01_to_0.02,daytime_core,187478,110413,2492,5,4,4984,4984,True
1,0.01_to_0.02,evening_transition,36136,24334,2092,5,4,4184,4184,True
2,0.01_to_0.02,morning_transition,22307,16036,1939,5,4,3878,3878,True
3,0.01_to_0.02,overnight,24617,18694,1963,5,4,3926,3926,True
4,0.02_to_0.05,daytime_core,240085,129869,2531,5,4,5062,5062,True
5,0.02_to_0.05,evening_transition,45297,28449,2144,5,4,4288,4288,True
6,0.02_to_0.05,morning_transition,28125,19190,2040,5,4,4080,4080,True
7,0.02_to_0.05,overnight,32202,23045,2057,5,4,4114,4114,True
8,0.05_to_0.10,daytime_core,182420,109291,2434,5,4,4868,4868,True
9,0.05_to_0.10,evening_transition,32276,22166,2010,5,4,4020,4020,True


VEHICLE


,vehicle_confidence_band,time_window,eligible_images,unique_sequences,unique_sites,geographic_groups,seasons,site_cap_capacity,estimated_candidate_capacity,meets_candidate_target
0,0.01_to_0.02,daytime_core,70474,44991,1058,5,4,2116,2116,True
1,0.01_to_0.02,evening_transition,13474,9035,737,5,4,1474,1474,True
2,0.01_to_0.02,morning_transition,10804,7431,696,5,4,1392,1392,True
3,0.01_to_0.02,overnight,8962,5856,446,5,4,892,892,True
4,0.02_to_0.05,daytime_core,74627,44382,1012,5,4,2024,2024,True
5,0.02_to_0.05,evening_transition,13965,8929,691,5,4,1382,1382,True
6,0.02_to_0.05,morning_transition,11583,7454,612,5,4,1224,1224,True
7,0.02_to_0.05,overnight,8129,5381,393,5,4,786,786,True
8,0.05_to_0.10,daytime_core,47180,29764,884,5,4,1768,1768,True
9,0.05_to_0.10,evening_transition,8870,6070,525,5,4,1050,1050,True


In [11]:
cell_feasibility_summary = []

for target_class in TARGET_CLASSES:
    cell_table = cell_availability_tables[target_class]

    cell_feasibility_summary.append({
        "target_class": target_class,
        "expected_cells": (
            len(POSITIVE_CONFIDENCE_BANDS)
            * len(USABLE_TIME_WINDOWS)
        ),
        "observed_cells": len(cell_table),
        "cells_meeting_target": int(
            cell_table["meets_candidate_target"].sum()
        ),
        "cells_below_target": int(
            (~cell_table["meets_candidate_target"]).sum()
        ),
        "minimum_eligible_images": int(
            cell_table["eligible_images"].min()
        ),
        "minimum_unique_sequences": int(
            cell_table["unique_sequences"].min()
        ),
        "minimum_unique_sites": int(
            cell_table["unique_sites"].min()
        ),
        "minimum_estimated_capacity": int(
            cell_table["estimated_candidate_capacity"].min()
        )
    })

cell_feasibility_summary = pd.DataFrame(
    cell_feasibility_summary
)

display(cell_feasibility_summary)

,target_class,expected_cells,observed_cells,cells_meeting_target,cells_below_target,minimum_eligible_images,minimum_unique_sequences,minimum_unique_sites,minimum_estimated_capacity
0,human,28,28,28,0,786,274,53,106
1,animal,28,28,28,0,20863,15324,1902,3804
2,vehicle,28,28,28,0,838,528,49,98


## Candidate selection

Positive candidate samples are constructed separately for human, animal and
vehicle detections using the same reusable sampling procedure.

Each class is sampled separately because the classes have different confidence
scores, confidence bands and bounding boxes. A photograph may therefore be
selected for more than one target class.

Within each class:

- each confidence-band × time-window cell has a target of 40 candidates;
- no more than one image is selected from the same sequence;
- no more than two candidate images are selected from the same site;
- geographic group and season are used to encourage diversity;
- constrained sampling cells are processed first.

In [12]:
selection_working_columns = [
    "photo_id",
    "sequence_id",
    "sequence_num",
    "site_id",
    "geographic_group",
    "season"
]

def get_positive_selection_pool(
    target_class,
    confidence_band,
    time_window
):
    band_column = sampling_columns[target_class]["band"]
    eligibility_mask = positive_eligibility_masks[target_class]

    cell_mask = (
        eligibility_mask
        & df[band_column].eq(confidence_band)
        & df["time_window"].eq(time_window)
    )

    cell_pool = df.loc[
        cell_mask,
        selection_working_columns
    ].copy()

    cell_pool.insert(
        0,
        "source_row_index",
        cell_pool.index
    )

    cell_pool.reset_index(
        drop=True,
        inplace=True
    )

    cell_pool["target_class"] = target_class
    cell_pool["sampling_confidence_band"] = confidence_band
    cell_pool["sampling_time_window"] = time_window

    return cell_pool

In [13]:
def select_candidates_from_cell(
    cell_pool,
    n_candidates,
    used_sequences=None,
    site_counts=None,
    site_cap=2,
    random_seed=42
):
    if used_sequences is None:
        used_sequences = set()

    if site_counts is None:
        site_counts = Counter()

    pool = cell_pool.copy()

    pool["balance_group"] = (
        pool["geographic_group"].astype(str)
        + " | "
        + pool["season"].astype(str)
    )

    rng = np.random.default_rng(random_seed)

    balance_groups = (
        pool["balance_group"]
        .drop_duplicates()
        .tolist()
    )

    rng.shuffle(balance_groups)

    group_queues = {}
    group_positions = {}

    for group_number, balance_group in enumerate(balance_groups):
        group_indices = pool.index[
            pool["balance_group"].eq(balance_group)
        ].to_numpy()

        group_rng = np.random.default_rng(
            random_seed + group_number + 1
        )

        group_rng.shuffle(group_indices)

        group_queues[balance_group] = group_indices
        group_positions[balance_group] = 0

    selected_indices = []
    selection_rounds = []

    selection_round = 1

    while len(selected_indices) < n_candidates:
        active_groups = [
            balance_group
            for balance_group in balance_groups
            if group_positions[balance_group]
            < len(group_queues[balance_group])
        ]

        if not active_groups:
            break

        rng.shuffle(active_groups)

        selections_this_round = 0

        for balance_group in active_groups:
            queue = group_queues[balance_group]
            position = group_positions[balance_group]

            selected_from_group = False

            while position < len(queue):
                row_index = queue[position]
                position += 1

                sequence_id = pool.at[
                    row_index,
                    "sequence_id"
                ]

                site_id = pool.at[
                    row_index,
                    "site_id"
                ]

                if sequence_id in used_sequences:
                    continue

                if site_counts[site_id] >= site_cap:
                    continue

                selected_indices.append(row_index)
                selection_rounds.append(selection_round)

                used_sequences.add(sequence_id)
                site_counts[site_id] += 1

                selections_this_round += 1
                selected_from_group = True
                break

            group_positions[balance_group] = position

            if len(selected_indices) >= n_candidates:
                break

        if selections_this_round == 0:
            break

        selection_round += 1

    selected = pool.loc[
        selected_indices
    ].copy()

    selected["selection_round"] = selection_rounds

    selected["candidate_rank_in_cell"] = np.arange(
        1,
        len(selected) + 1
    )

    selected.drop(
        columns="balance_group",
        inplace=True
    )

    diagnostics = {
        "available_images": len(pool),
        "requested_candidates": n_candidates,
        "selected_candidates": len(selected),
        "unique_sequences_selected": (
            selected["sequence_id"].nunique()
        ),
        "unique_sites_selected": (
            selected["site_id"].nunique()
        ),
        "maximum_images_from_one_site": (
            selected["site_id"].value_counts().max()
            if len(selected) > 0
            else 0
        ),
        "geographic_groups_selected": (
            selected["geographic_group"].nunique()
        ),
        "seasons_selected": (
            selected["season"].nunique()
        )
    }

    return (
        selected,
        diagnostics,
        used_sequences,
        site_counts
    )

## Full positive candidate sampling

Candidate images are selected separately for human, animal and vehicle
detections.

Within each target class, sampling cells are processed from the lowest to the
highest estimated capacity. This gives the most constrained cells priority
before globally applying the sequence and site limits.

For each class:

- 40 candidates are requested from every confidence-band × time-window cell;
- no sequence may contribute more than one candidate;
- no site may contribute more than two candidates;
- geography and season are used to encourage diversity;
- a separate random seed is used for each sampling cell.

In [14]:
class_seed_offsets = {
    "human": 0,
    "animal": 10_000,
    "vehicle": 20_000
}

positive_candidate_samples = {}
positive_sampling_diagnostics = []

for target_class in TARGET_CLASSES:
    band_column = sampling_columns[target_class]["band"]

    used_sequences = set()
    site_counts = Counter()
    selected_cells = []

    cell_order = (
        cell_availability_tables[target_class]
        .sort_values(
            [
                "estimated_candidate_capacity",
                band_column,
                "time_window"
            ]
        )
        .reset_index(drop=True)
    )

    print("Sampling class:", target_class)
    print("Cells to process:", len(cell_order))

    for cell_position, cell_row in cell_order.iterrows():
        confidence_band = cell_row[band_column]
        time_window = cell_row["time_window"]

        cell_pool = get_positive_selection_pool(
            target_class=target_class,
            confidence_band=confidence_band,
            time_window=time_window
        )

        cell_seed = (
            RANDOM_SEED
            + class_seed_offsets[target_class]
            + cell_position
        )

        (
            selected,
            diagnostics,
            used_sequences,
            site_counts
        ) = select_candidates_from_cell(
            cell_pool=cell_pool,
            n_candidates=CANDIDATES_PER_CELL,
            used_sequences=used_sequences,
            site_counts=site_counts,
            site_cap=CANDIDATE_SITE_CAP,
            random_seed=cell_seed
        )

        selected["cell_processing_order"] = (
            cell_position + 1
        )

        selected_cells.append(selected)

        diagnostics.update({
            "target_class": target_class,
            "confidence_band": confidence_band,
            "time_window": time_window,
            "cell_processing_order": cell_position + 1,
            "random_seed": cell_seed,
            "total_used_sequences_after_cell": len(
                used_sequences
            ),
            "total_used_sites_after_cell": len(
                site_counts
            )
        })

        positive_sampling_diagnostics.append(
            diagnostics
        )

    class_candidates = pd.concat(
        selected_cells,
        ignore_index=True
    )

    positive_candidate_samples[target_class] = (
        class_candidates
    )

    print("Selected candidates:", len(class_candidates))
    print("Unique sequences:", class_candidates["sequence_id"].nunique())
    print("Unique sites:", class_candidates["site_id"].nunique())
    print(
        "Maximum candidates from one site:",
        class_candidates["site_id"].value_counts().max()
    )

Sampling class: human
Cells to process: 28
Selected candidates: 1120
Unique sequences: 1120
Unique sites: 721
Maximum candidates from one site: 2
Sampling class: animal
Cells to process: 28
Selected candidates: 1120
Unique sequences: 1120
Unique sites: 782
Maximum candidates from one site: 2
Sampling class: vehicle
Cells to process: 28
Selected candidates: 1120
Unique sequences: 1120
Unique sites: 681
Maximum candidates from one site: 2


In [15]:
positive_sampling_diagnostics = pd.DataFrame(
    positive_sampling_diagnostics
)

display(
    positive_sampling_diagnostics[
        [
            "target_class",
            "confidence_band",
            "time_window",
            "available_images",
            "requested_candidates",
            "selected_candidates",
            "unique_sequences_selected",
            "unique_sites_selected",
            "maximum_images_from_one_site",
            "geographic_groups_selected",
            "seasons_selected",
            "cell_processing_order"
        ]
    ]
)

,target_class,confidence_band,time_window,available_images,requested_candidates,selected_candidates,unique_sequences_selected,unique_sites_selected,maximum_images_from_one_site,geographic_groups_selected,seasons_selected,cell_processing_order
0,human,0.90_to_1.00,overnight,786,40,40,40,29,2,4,4,1
1,human,0.70_to_0.90,overnight,1823,40,40,40,33,2,4,4,2
2,human,0.30_to_0.70,overnight,1114,40,40,40,35,2,4,4,3
3,human,0.10_to_0.30,overnight,829,40,40,40,38,2,4,4,4
4,human,0.90_to_1.00,morning_transition,2770,40,40,40,32,2,5,4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
79,vehicle,0.01_to_0.02,evening_transition,13474,40,40,40,39,2,5,4,24
80,vehicle,0.10_to_0.30,daytime_core,77512,40,40,40,37,2,5,4,25
81,vehicle,0.05_to_0.10,daytime_core,47180,40,40,40,38,2,4,4,26
82,vehicle,0.02_to_0.05,daytime_core,74627,40,40,40,39,2,5,4,27


In [16]:
positive_candidate_summary = []

for target_class in TARGET_CLASSES:
    candidates = positive_candidate_samples[
        target_class
    ]

    positive_candidate_summary.append({
        "target_class": target_class,
        "candidate_records": len(candidates),
        "expected_candidate_records": (
            len(POSITIVE_CONFIDENCE_BANDS)
            * len(USABLE_TIME_WINDOWS)
            * CANDIDATES_PER_CELL
        ),
        "sampling_cells": (
            candidates[
                [
                    "sampling_confidence_band",
                    "sampling_time_window"
                ]
            ]
            .drop_duplicates()
            .shape[0]
        ),
        "unique_photos": candidates["photo_id"].nunique(),
        "unique_sequences": candidates[
            "sequence_id"
        ].nunique(),
        "unique_sites": candidates["site_id"].nunique(),
        "maximum_candidates_per_site": (
            candidates["site_id"]
            .value_counts()
            .max()
        ),
        "geographic_groups": candidates[
            "geographic_group"
        ].nunique(),
        "seasons": candidates["season"].nunique()
    })

positive_candidate_summary = pd.DataFrame(
    positive_candidate_summary
)

display(positive_candidate_summary)

,target_class,candidate_records,expected_candidate_records,sampling_cells,unique_photos,unique_sequences,unique_sites,maximum_candidates_per_site,geographic_groups,seasons
0,human,1120,1120,28,1120,1120,721,2,5,4
1,animal,1120,1120,28,1120,1120,782,2,5,4
2,vehicle,1120,1120,28,1120,1120,681,2,5,4


In [17]:
cell_candidate_counts = (
    pd.concat(
        positive_candidate_samples.values(),
        ignore_index=True
    )
    .groupby(
        [
            "target_class",
            "sampling_confidence_band",
            "sampling_time_window"
        ],
        observed=True
    )
    .size()
    .rename("selected_candidates")
    .reset_index()
)

display(cell_candidate_counts)

,target_class,sampling_confidence_band,sampling_time_window,selected_candidates
0,animal,0.01_to_0.02,daytime_core,40
1,animal,0.01_to_0.02,evening_transition,40
2,animal,0.01_to_0.02,morning_transition,40
3,animal,0.01_to_0.02,overnight,40
4,animal,0.02_to_0.05,daytime_core,40
...,...,...,...,...
79,vehicle,0.70_to_0.90,overnight,40
80,vehicle,0.90_to_1.00,daytime_core,40
81,vehicle,0.90_to_1.00,evening_transition,40
82,vehicle,0.90_to_1.00,morning_transition,40


## Positive candidate records and unique image queue

The class-specific candidate samples are combined into a single table of
candidate class-records.

Because one photograph may be selected independently for more than one target
class, a second table containing one row per unique photograph is created for
image retrieval and annotation.

The `contains_human` field is deliberately excluded from these outputs to
avoid influencing manual annotation.

In [20]:
class_prefixes = {
    "human": "HUM",
    "animal": "ANI",
    "vehicle": "VEH"
}

positive_candidate_records = pd.concat(
    [
        positive_candidate_samples[target_class]
        for target_class in TARGET_CLASSES
    ],
    ignore_index=True
)

positive_candidate_records["target_class"] = pd.Categorical(
    positive_candidate_records["target_class"],
    categories=TARGET_CLASSES,
    ordered=True
)

positive_candidate_records = (
    positive_candidate_records
    .sort_values(
        [
            "target_class",
            "cell_processing_order",
            "candidate_rank_in_cell"
        ]
    )
    .reset_index(drop=True)
)

positive_candidate_records["class_candidate_number"] = (
    positive_candidate_records
    .groupby("target_class", observed=True)
    .cumcount()
    + 1
)

positive_candidate_records["candidate_record_id"] = [
    (
        f"POS_{class_prefixes[str(target_class)]}_"
        f"{candidate_number:04d}"
    )
    for target_class, candidate_number in zip(
        positive_candidate_records["target_class"],
        positive_candidate_records["class_candidate_number"]
    )
]

positive_candidate_records["target_class"] = (
    positive_candidate_records["target_class"].astype(str)
)

positive_candidate_records["sampling_stage"] = (
    "positive_candidate_pool"
)

print(
    "Positive candidate class-records:",
    f"{len(positive_candidate_records):,}"
)

print(
    "Unique candidate record IDs:",
    positive_candidate_records[
        "candidate_record_id"
    ].nunique()
)

Positive candidate class-records: 3,360
Unique candidate record IDs: 3360


In [21]:
candidate_metadata_columns = [
    "photo_id",
    "sequence_id",
    "sequence_num",
    "site_id",
    "site_name",
    "filename",
    "dirname",
    "taken",
    "taken_dt",
    "year",
    "month",
    "hour",
    "date",
    "season",
    "time_window",
    "latitude",
    "longitude",
    "geographic_group",
    "missing_from_supp",
    "missing_positive_from_supp",

    "mega_human_confidence",
    "mega_animal_confidence",
    "mega_vehicle_confidence",

    "human_confidence_band",
    "animal_confidence_band",
    "vehicle_confidence_band",

    "n_human_boxes",
    "n_animal_boxes",
    "n_vehicle_boxes",

    "human_bbox_prob",
    "animal_bbox_prob",
    "vehicle_bbox_prob",

    "human_xmin",
    "human_ymin",
    "human_xmax",
    "human_ymax",

    "animal_xmin",
    "animal_ymin",
    "animal_xmax",
    "animal_ymax",

    "vehicle_xmin",
    "vehicle_ymin",
    "vehicle_xmax",
    "vehicle_ymax",

    "human_bbox_width_px",
    "human_bbox_height_px",
    "human_bbox_area_px",

    "animal_bbox_width_px",
    "animal_bbox_height_px",
    "animal_bbox_area_px",

    "vehicle_bbox_width_px",
    "vehicle_bbox_height_px",
    "vehicle_bbox_area_px",

    "human_has_bbox",
    "animal_has_bbox",
    "vehicle_has_bbox"
]

missing_metadata_columns = [
    column
    for column in candidate_metadata_columns
    if column not in df.columns
]

print("Missing metadata columns:", missing_metadata_columns)

assert not missing_metadata_columns

Missing metadata columns: []


In [22]:
source_indices = (
    positive_candidate_records["source_row_index"]
    .astype(int)
    .to_numpy()
)

selected_metadata = (
    df.loc[
        source_indices,
        candidate_metadata_columns
    ]
    .reset_index(drop=True)
)

assert np.array_equal(
    positive_candidate_records["photo_id"].to_numpy(),
    selected_metadata["photo_id"].to_numpy()
)

new_metadata_columns = [
    column
    for column in candidate_metadata_columns
    if column not in positive_candidate_records.columns
]

positive_candidate_records = pd.concat(
    [
        positive_candidate_records.reset_index(drop=True),
        selected_metadata[new_metadata_columns]
    ],
    axis=1
)

print(
    "Combined candidate table shape:",
    positive_candidate_records.shape
)

Combined candidate table shape: (3360, 66)


In [24]:
photo_class_membership = (
    positive_candidate_records
    .groupby("photo_id", observed=True)
    .agg(
        number_of_target_classes=(
            "target_class",
            "nunique"
        ),
        selected_for_classes=(
            "target_class",
            lambda values: "|".join(
                sorted(set(values))
            )
        )
    )
    .reset_index()
)

overlap_summary = (
    photo_class_membership[
        "number_of_target_classes"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_target_classes")
    .reset_index(name="unique_images")
)

display(overlap_summary)

print(
    "Total class-records:",
    f"{len(positive_candidate_records):,}"
)

print(
    "Unique candidate images:",
    f"{photo_class_membership['photo_id'].nunique():,}"
)

print(
    "Images selected for multiple classes:",
    f"{(
        photo_class_membership[
            'number_of_target_classes'
        ] > 1
    ).sum():,}"
)

,number_of_target_classes,unique_images
0,1,3340
1,2,10


Total class-records: 3,360
Unique candidate images: 3,350
Images selected for multiple classes: 10


In [25]:
queue_metadata_columns = [
    "source_row_index",
    *candidate_metadata_columns
]

positive_candidate_image_queue = (
    positive_candidate_records[
        queue_metadata_columns
    ]
    .drop_duplicates("photo_id")
    .copy()
)

positive_candidate_image_queue = (
    positive_candidate_image_queue
    .merge(
        photo_class_membership,
        on="photo_id",
        how="left",
        validate="one_to_one"
    )
)

In [26]:
for target_class in TARGET_CLASSES:
    class_records = (
        positive_candidate_records.loc[
            positive_candidate_records[
                "target_class"
            ].eq(target_class)
        ]
        .set_index("photo_id")
    )

    positive_candidate_image_queue[
        f"candidate_for_{target_class}"
    ] = (
        positive_candidate_image_queue["photo_id"]
        .isin(class_records.index)
    )

    positive_candidate_image_queue[
        f"{target_class}_candidate_record_id"
    ] = (
        positive_candidate_image_queue["photo_id"]
        .map(class_records["candidate_record_id"])
    )

    positive_candidate_image_queue[
        f"{target_class}_sampling_confidence_band"
    ] = (
        positive_candidate_image_queue["photo_id"]
        .map(
            class_records[
                "sampling_confidence_band"
            ]
        )
    )

    positive_candidate_image_queue[
        f"{target_class}_sampling_time_window"
    ] = (
        positive_candidate_image_queue["photo_id"]
        .map(
            class_records[
                "sampling_time_window"
            ]
        )
    )

In [27]:
positive_candidate_image_queue[
    "image_access_status"
] = "pending"

positive_candidate_image_queue[
    "image_access_notes"
] = ""

positive_candidate_image_queue[
    "image_width_px"
] = pd.NA

positive_candidate_image_queue[
    "image_height_px"
] = pd.NA

positive_candidate_image_queue = (
    positive_candidate_image_queue
    .sort_values("photo_id")
    .reset_index(drop=True)
)

print(
    "Unique image queue rows:",
    f"{len(positive_candidate_image_queue):,}"
)

print(
    "Unique photo IDs:",
    positive_candidate_image_queue[
        "photo_id"
    ].nunique()
)

assert positive_candidate_image_queue[
    "photo_id"
].is_unique

Unique image queue rows: 3,350
Unique photo IDs: 3350


In [28]:
candidate_records_path = (
    OUTPUT_DIR
    / "positive_candidate_records.csv"
)

image_queue_path = (
    OUTPUT_DIR
    / "positive_candidate_image_queue.csv"
)

diagnostics_path = (
    OUTPUT_DIR
    / "positive_sampling_diagnostics.csv"
)

summary_path = (
    OUTPUT_DIR
    / "positive_candidate_summary.csv"
)

cell_counts_path = (
    OUTPUT_DIR
    / "positive_cell_candidate_counts.csv"
)

positive_candidate_records.to_csv(
    candidate_records_path,
    index=False
)

positive_candidate_image_queue.to_csv(
    image_queue_path,
    index=False
)

positive_sampling_diagnostics.to_csv(
    diagnostics_path,
    index=False
)

positive_candidate_summary.to_csv(
    summary_path,
    index=False
)

cell_candidate_counts.to_csv(
    cell_counts_path,
    index=False
)

print("Saved:", candidate_records_path)
print("Saved:", image_queue_path)
print("Saved:", diagnostics_path)
print("Saved:", summary_path)
print("Saved:", cell_counts_path)

Saved: sampling_outputs\positive_candidate_records.csv
Saved: sampling_outputs\positive_candidate_image_queue.csv
Saved: sampling_outputs\positive_sampling_diagnostics.csv
Saved: sampling_outputs\positive_candidate_summary.csv
Saved: sampling_outputs\positive_cell_candidate_counts.csv
